# Step 6: The 1D Transverse-Field Ising Model

The Transverse-Field Ising Model (TFIM) is the simplest quantum model with a phase transition. It has two competing terms whose balance drives a **quantum phase transition** at zero temperature — not controlled by thermal fluctuations, but by quantum ones.

In notebook 05 we built the ED machinery for the Heisenberg model. Here we adapt it to the TFIM, where the structure of the Hamiltonian — one diagonal part and one off-diagonal part — makes both the physics and the numerics especially transparent.

**What you will do:**
- Build the TFIM Hamiltonian as a sparse matrix from $\hat{\sigma}^z$ and $\hat{\sigma}^x$ terms
- Verify the ordered ($\Gamma = 0$) and disordered ($J = 0$) limits analytically
- Sweep $\Gamma/J$ and observe the energy gap close at the quantum critical point $\Gamma_c = J$
- Track the magnetisation order parameter and its vanishing at the transition
- Study how the gap closes as $\Delta \sim 1/N$ at criticality

**Prerequisites:** Notebook 05 (ED machinery, binary basis encoding).

## 6.1  The Hamiltonian and Its Two Limits

The 1D TFIM on a chain of $N$ sites (open boundary conditions) is:

$$\hat{H} = -J \sum_{i=1}^{N-1} \hat{\sigma}^z_i \hat{\sigma}^z_{i+1} - \Gamma \sum_{i=1}^{N} \hat{\sigma}^x_i$$

where $\hat{\sigma}^z = \begin{pmatrix}1&0\\0&-1\end{pmatrix}$ and $\hat{\sigma}^x = \begin{pmatrix}0&1\\1&0\end{pmatrix}$ are the Pauli matrices (note: these are twice the spin-$1/2$ operators $\hat{S}^\alpha$, so their eigenvalues are $\pm 1$, not $\pm 1/2$).

**Limit 1: $\Gamma = 0$.** The Hamiltonian is $\hat{H} = -J\sum_i \hat{\sigma}^z_i \hat{\sigma}^z_{i+1}$, diagonal in the $z$-basis. With $J > 0$, the two degenerate ground states are $|{\uparrow\uparrow\cdots\uparrow}\rangle$ and $|{\downarrow\downarrow\cdots\downarrow}\rangle$, both with energy $-J(N-1)$. The system is **ordered**.

**Limit 2: $J = 0$.** The Hamiltonian is $\hat{H} = -\Gamma\sum_i \hat{\sigma}^x_i$. The eigenstates of $\hat{\sigma}^x$ are $|{+}\rangle = (|{\uparrow}\rangle + |{\downarrow}\rangle)/\sqrt{2}$ and $|{-}\rangle = (|{\uparrow}\rangle - |{\downarrow}\rangle)/\sqrt{2}$. The unique ground state is $|{+}\rangle^{\otimes N}$ with energy $-\Gamma N$. The system is **disordered** — a product state with no spin correlations in the $z$-direction.

As $\Gamma/J$ is swept from $0$ to $\infty$, a quantum phase transition occurs at $\Gamma_c = J$.

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt

# Pauli matrices (2x2)
sigma_z = np.array([[ 1,  0], [ 0, -1]], dtype=float)
sigma_x = np.array([[ 0,  1], [ 1,  0]], dtype=float)
sigma_y = np.array([[ 0, -1j], [1j,  0]], dtype=complex)
I2      = np.eye(2)

# ── Verify the two limits analytically ──────────────────────────────────────
# Limit 1: Gamma=0, N=4, J=1.  E0 = -J*(N-1) = -3,  doubly degenerate.
N = 4
H_ZZ = np.zeros((2**N, 2**N))
for i in range(N - 1):
    # Build sigma_z_i * sigma_z_{i+1} by tensor product
    op = 1.0
    for k in range(N):
        op = np.kron(op, sigma_z if k in (i, i+1) else I2)
    H_ZZ -= op   # J = 1

e_limit1 = np.sort(np.linalg.eigh(H_ZZ)[0])
print("Limit 1 (Gamma=0, N=4, J=1):")
print(f"  5 lowest eigenvalues: {e_limit1[:5]}")
print(f"  Expected E0 = -{N-1}.0  (doubly degenerate)")
print()

# Limit 2: J=0, N=4, Gamma=1.  E0 = -Gamma*N = -4,  unique ground state.
H_X = np.zeros((2**N, 2**N))
for i in range(N):
    op = 1.0
    for k in range(N):
        op = np.kron(op, sigma_x if k == i else I2)
    H_X -= op   # Gamma = 1

e_limit2 = np.sort(np.linalg.eigh(H_X)[0])
print("Limit 2 (J=0, N=4, Gamma=1):")
print(f"  5 lowest eigenvalues: {e_limit2[:5]}")
print(f"  Expected E0 = -{N}.0  (unique ground state, gap = 2)")

## 6.2  Building the TFIM Hamiltonian as a Sparse Matrix

For large $N$, the tensor-product construction above is expensive (builds dense $2^N \times 2^N$ matrices). We use the same bit-operation approach from notebook 05 to build a sparse CSR matrix directly.

The two parts of the Hamiltonian have simple action on the binary basis $|s\rangle$:

- **$-J\hat{\sigma}^z_i\hat{\sigma}^z_{i+1}$** is diagonal: it contributes $-J \cdot (2b_i - 1)(2b_j - 1)$ to $H_{ss}$, where $b_k \in \{0, 1\}$ is the bit at site $k$.

- **$-\Gamma\hat{\sigma}^x_i$** is off-diagonal: $\hat{\sigma}^x_i|s\rangle = |s \oplus e_i\rangle$ (flip bit $i$). It contributes $-\Gamma$ to $H_{s', s}$ where $s' = s \oplus (1 \ll i)$.

In [ ]:
def build_tfim(N, J=1.0, Gamma=1.0, pbc=False):
    """
    Build the 1D TFIM Hamiltonian as a sparse CSR matrix.
    H = -J sum_{i} sigma_z_i sigma_z_{i+1}  -  Gamma sum_{i} sigma_x_i

    Basis: integers 0..2^N-1, bit i = 1 means spin-up (sigma_z eigenvalue +1).
    pbc: if True, add bond (N-1, 0) for periodic boundaries.
    """
    dim    = 2 ** N
    states = np.arange(dim)
    rows, cols, vals = [], [], []

    # ── Ising coupling: -J * sigma_z_i * sigma_z_{i+1} (diagonal) ────────────
    n_bonds = N if pbc else N - 1
    for i in range(n_bonds):
        j    = (i + 1) % N
        bi   = (states >> i) & 1          # 0 or 1
        bj   = (states >> j) & 1
        sz_i = 2 * bi.astype(float) - 1  # +1 or -1  (Pauli eigenvalue)
        sz_j = 2 * bj.astype(float) - 1
        rows.extend(states.tolist())
        cols.extend(states.tolist())
        vals.extend((-J * sz_i * sz_j).tolist())

    # ── Transverse field: -Gamma * sigma_x_i (off-diagonal, bit flip) ────────
    for i in range(N):
        s_flip = states ^ (1 << i)        # flip bit i
        rows.extend(states.tolist())
        cols.extend(s_flip.tolist())
        vals.extend([-Gamma] * dim)

    return csr_matrix((vals, (rows, cols)), shape=(dim, dim), dtype=float)


# ── Cross-check against the explicit tensor-product result above ──────────────
J, Gamma_test, N_check = 1.0, 0.5, 4
H_sparse = build_tfim(N_check, J=J, Gamma=Gamma_test, pbc=False)
H_dense  = H_ZZ + Gamma_test * H_X   # reusing the matrices built above (J=1, note sign)
# Rebuild H_X with Gamma_test for exact comparison
H_X_check = np.zeros((2**N_check, 2**N_check))
for i in range(N_check):
    op = 1.0
    for k in range(N_check):
        op = np.kron(op, sigma_x if k == i else I2)
    H_X_check -= Gamma_test * op

H_ref = H_ZZ + H_X_check

diff = np.max(np.abs(H_sparse.toarray() - H_ref))
print(f"Max element difference (sparse vs dense tensor product): {diff:.2e}")
print(f"Matrices agree: {diff < 1e-12}")
print()

# Verify the two limits again with the sparse builder
N_v = 8
e_ordered    = eigsh(build_tfim(N_v, J=1.0, Gamma=0.0), k=3, which='SA',
                     return_eigenvectors=False)
e_disordered = eigsh(build_tfim(N_v, J=0.0, Gamma=1.0), k=3, which='SA',
                     return_eigenvectors=False)
print(f"Ordered limit   (Gamma=0, N={N_v}): 3 lowest = {np.sort(e_ordered)}")
print(f"  Expected E0 = -{N_v-1}.0  (quasi-degenerate pair at bottom)")
print(f"Disordered limit (J=0, N={N_v}):   3 lowest = {np.sort(e_disordered)}")
print(f"  Expected E0 = -{N_v}.0  (unique, gap = 2)")

## 6.3  Energy Spectrum vs $\Gamma / J$

We now sweep $\Gamma/J$ from $0$ to $2$ and track the four lowest eigenvalues. Three qualitatively different regimes are visible:

- **Ordered phase** ($\Gamma < J$): The two lowest states form a quasi-degenerate pair, split by a gap $\Delta_1 \sim e^{-N/\xi}$ that is exponentially small in system size. This splitting reflects quantum tunnelling between the two ordered ground states $|{\uparrow\cdots\uparrow}\rangle$ and $|{\downarrow\cdots\downarrow}\rangle$.

- **Critical point** ($\Gamma = J$): The gap between the ground state and first excited state is minimised, closing as $\Delta \sim 1/N$ as $N \to \infty$.

- **Disordered phase** ($\Gamma > J$): The ground state is unique and the gap is of order $|\Gamma - J|$, growing linearly away from the critical point.

In [ ]:
N_spec = 12
J      = 1.0

Gamma_values = np.linspace(0.05, 2.0, 60)
n_eigs       = 4
spectra      = np.zeros((len(Gamma_values), n_eigs))

for idx, Gamma in enumerate(Gamma_values):
    H    = build_tfim(N_spec, J=J, Gamma=Gamma, pbc=False)
    evals = np.sort(eigsh(H, k=n_eigs, which='SA', return_eigenvectors=False))
    spectra[idx] = evals

# Normalise by N so energies per site are comparable
spectra_per_site = spectra / N_spec

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2166ac', '#d6604d', '#4dac26', '#8856a7']
labels = [r'$E_0$', r'$E_1$', r'$E_2$', r'$E_3$']

for k in range(n_eigs):
    axes[0].plot(Gamma_values / J, spectra_per_site[:, k],
                 lw=1.8, color=colors[k], label=labels[k])
axes[0].axvline(1.0, color='k', ls='--', lw=1.2, label=r'$\Gamma_c = J$')
axes[0].set_xlabel(r'$\Gamma / J$', fontsize=13)
axes[0].set_ylabel(r'$E_k / N$', fontsize=13)
axes[0].set_title(f'Energy spectrum  ($N = {N_spec}$)', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Right panel: gap E1 - E0
gap = spectra[:, 1] - spectra[:, 0]
axes[1].plot(Gamma_values / J, gap, lw=2, color='steelblue')
axes[1].axvline(1.0, color='k', ls='--', lw=1.2, label=r'$\Gamma_c = J$')
min_idx = np.argmin(gap)
axes[1].axvline(Gamma_values[min_idx] / J, color='tomato', ls=':', lw=1.5,
                label=f'Numerical min at $\\Gamma/J = {Gamma_values[min_idx]/J:.2f}$')
axes[1].set_xlabel(r'$\Gamma / J$', fontsize=13)
axes[1].set_ylabel(r'Gap $\Delta = E_1 - E_0$', fontsize=13)
axes[1].set_title(f'Energy gap  ($N = {N_spec}$)', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Gap at Gamma/J = 0.5: {gap[np.argmin(np.abs(Gamma_values - 0.5 * J))]:.4f}")
print(f"Gap at Gamma/J = 1.0: {gap[min_idx]:.4f}  (minimum)")
print(f"Gap at Gamma/J = 1.5: {gap[np.argmin(np.abs(Gamma_values - 1.5 * J))]:.4f}")

## 6.4  Order Parameter

The order parameter for the TFIM is the magnetisation in the $z$-direction:

$$m = \frac{1}{N} \sum_i \langle \hat{\sigma}^z_i \rangle$$

However, there is a subtlety. The TFIM Hamiltonian has a $\mathbb{Z}_2$ symmetry: flipping all spins $\hat{\sigma}^z_i \to -\hat{\sigma}^z_i$ leaves $H$ unchanged (since both $\hat{\sigma}^z_i\hat{\sigma}^z_{i+1}$ and $\hat{\sigma}^x_i$ are $\mathbb{Z}_2$-symmetric). This means the exact finite-size ground state is always a $\mathbb{Z}_2$-symmetric superposition, and $\langle \hat{\sigma}^z_i \rangle = 0$ exactly for all $\Gamma$.

In the ordered phase, what actually happens is that $E_0$ and $E_1$ become exponentially degenerate, and the physical ordered states are $|\psi_+\rangle \pm |\psi_-\rangle$ — the symmetric and antisymmetric combinations of the GHZ-like pair. We can diagnose this ordering without breaking symmetry by computing the **squared magnetisation**:

$$m^2 = \frac{1}{N^2} \left\langle \left( \sum_i \hat{\sigma}^z_i \right)^2 \right\rangle$$

This is non-zero in the ordered phase even for the symmetric ground state, because $\langle (\hat{\sigma}^z_{\rm tot})^2 \rangle \sim N^2$ there (long-range correlations), while in the disordered phase $\langle (\hat{\sigma}^z_{\rm tot})^2 \rangle \sim N$ (uncorrelated), so $m^2 \sim 1/N \to 0$.

In [ ]:
def sigma_z_total_squared_ev(psi0, N):
    """
    Compute <psi0 | (sum_i sigma_z_i)^2 | psi0>.

    sigma_z_i is diagonal: sigma_z_i |s> = (2*bit_i - 1) |s>.
    So sum_i sigma_z_i |s> = (number of up-bits - number of down-bits) |s>.
    """
    states   = np.arange(2 ** N)
    # Count up-bits for each basis state
    n_up     = np.array([bin(s).count('1') for s in states], dtype=float)
    sz_tot   = 2 * n_up - N   # sum of sigma_z eigenvalues: ranges from -N to +N
    return float(np.sum(np.abs(psi0) ** 2 * sz_tot ** 2))


N_op = 14
Gamma_scan = np.linspace(0.1, 2.0, 50)
m2_values  = []

for Gamma in Gamma_scan:
    H = build_tfim(N_op, J=J, Gamma=Gamma, pbc=False)
    _, evec = eigsh(H, k=1, which='SA')
    psi0    = evec[:, 0]
    m2      = sigma_z_total_squared_ev(psi0, N_op) / N_op ** 2
    m2_values.append(m2)

m2_values = np.array(m2_values)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(Gamma_scan / J, m2_values, lw=2, color='steelblue',
        label=r'$m^2 = \langle (\hat{\sigma}^z_{\rm tot})^2 \rangle / N^2$')
ax.axvline(1.0, color='k', ls='--', lw=1.2, label=r'$\Gamma_c = J$')
ax.set_xlabel(r'$\Gamma / J$', fontsize=13)
ax.set_ylabel(r'$m^2$', fontsize=13)
ax.set_title(f'Squared magnetisation  ($N = {N_op}$)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Show the finite-size quasi-degeneracy in the ordered phase
print("Energy splitting E1-E0 in the ordered phase (Gamma/J = 0.3):")
for N_split in [6, 8, 10, 12, 14, 16]:
    H      = build_tfim(N_split, J=J, Gamma=0.3 * J, pbc=False)
    evals  = np.sort(eigsh(H, k=2, which='SA', return_eigenvectors=False))
    split  = evals[1] - evals[0]
    print(f"  N = {N_split:>2}: E1 - E0 = {split:.2e}")

## 6.5  Gap Scaling at the Critical Point

At the quantum critical point $\Gamma_c = J$, the energy gap closes in the thermodynamic limit. For a finite chain, the gap scales as:

$$\Delta(N) \sim N^{-z}$$

where $z$ is the **dynamical critical exponent**. For the 1D TFIM, $z = 1$ (the dispersion is linear, like a relativistic system), so $\Delta \sim 1/N$.

Away from the critical point in the disordered phase ($\Gamma > J$), the gap converges to a finite value as $N \to \infty$: $\Delta \to 2(\Gamma - J)$ (to leading order). Tracking how quickly the gap saturates vs how it falls towards zero reveals the location of the critical point — this is the foundation of the finite-size scaling analysis in notebook 07.

In [ ]:
N_values = [6, 8, 10, 12, 14, 16, 18, 20]
# Gap at three values of Gamma/J
Gamma_list = [0.6 * J, 1.0 * J, 1.4 * J]
label_list = [r'$\Gamma/J = 0.6$ (ordered)', r'$\Gamma/J = 1.0$ (critical)',
              r'$\Gamma/J = 1.4$ (disordered)']
color_list = ['#2166ac', '#d6604d', '#4dac26']

gap_data = {G: [] for G in Gamma_list}

for N in N_values:
    for G in Gamma_list:
        H    = build_tfim(N, J=J, Gamma=G, pbc=False)
        evals = np.sort(eigsh(H, k=3, which='SA', return_eigenvectors=False))
        # In ordered phase E0 ≈ E1 (quasi-degenerate); use E2-E0 to get the true gap
        # across both phases use E1-E0 which captures quasi-degeneracy too
        gap_data[G].append(evals[1] - evals[0])

# Fit the critical gap to Δ ~ N^α
gaps_crit = np.array(gap_data[1.0 * J])
log_N     = np.log(N_values)
log_gap   = np.log(gaps_crit)
slope_crit, intercept_crit = np.polyfit(log_N, log_gap, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for G, lbl, col in zip(Gamma_list, label_list, color_list):
    axes[0].plot(N_values, gap_data[G], 'o-', ms=6, lw=1.8, color=col, label=lbl)
axes[0].set_xlabel('System size $N$', fontsize=13)
axes[0].set_ylabel(r'Gap $\Delta = E_1 - E_0$', fontsize=13)
axes[0].set_title('Gap vs $N$ for three values of $\\Gamma / J$', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Log-log for critical gap
N_fit = np.linspace(min(N_values) * 0.9, max(N_values) * 1.1, 100)
axes[1].loglog(N_values, gaps_crit, 'o', ms=7, color='#d6604d', label='ED at $\\Gamma = J$')
axes[1].loglog(N_fit, np.exp(intercept_crit) * N_fit ** slope_crit, 'k--', lw=1.5,
               label=f'Fit: $\\Delta \\sim N^{{{slope_crit:.2f}}}$  (expected $-1$)')
axes[1].set_xlabel('System size $N$', fontsize=13)
axes[1].set_ylabel(r'Gap $\Delta$', fontsize=13)
axes[1].set_title(r'Gap at criticality: $\Delta \sim N^{-z}$, $z = 1$', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print(f"Critical gap scaling exponent: {slope_crit:.3f}  (expected z = -1)")
print()
print("Gap values at Gamma/J = 1.4 for increasing N (should saturate):")
for N, g in zip(N_values, gap_data[1.4 * J]):
    print(f"  N = {N:>2}: Delta = {g:.6f}")

## 6.6  What Makes This a Quantum Phase Transition?

The transition we have just observed has the same mathematical fingerprints as the classical Ising transition in notebook 01 — diverging correlation length, gap closing, order parameter vanishing — but its physical mechanism is entirely different.

| | Classical Ising | Quantum TFIM |
|---|---|---|
| Control parameter | Temperature $T$ | Transverse field $\Gamma$ |
| Fluctuations | Thermal ($k_B T$) | Quantum (zero-point) |
| Occurs at | $T = T_c > 0$ | $\Gamma = \Gamma_c$, $T = 0$ |
| Order parameter | $\langle \sigma^z \rangle$ (thermal avg) | $\langle \hat{\sigma}^z \rangle$ (ground state) |
| Relevant states | Thermal Boltzmann ensemble | Ground state wavefunction |

The transverse field $\Gamma$ induces **quantum tunnelling** between spin-up and spin-down. When $\Gamma$ is small, the tunnelling rate is low and the spins lock into a ferromagnetic configuration. When $\Gamma$ is large, tunnelling is rapid, and the spins are better described by eigenstates of $\hat{\sigma}^x$ — no $z$-magnetisation. At $\Gamma_c = J$, these two tendencies balance and the system is quantum critical.

### The Quantum-to-Classical Mapping

There is a deep correspondence between the 1D TFIM at $T = 0$ and the 2D classical Ising model. Writing the partition function $Z = \mathrm{Tr}\, e^{-\beta \hat{H}}$ as a path integral using the Suzuki-Trotter decomposition introduces an imaginary-time direction of extent $\beta$. At $T = 0$ ($\beta \to \infty$), this imaginary-time direction becomes infinite, and the resulting effective model is exactly the 2D classical Ising model on a $(N \times \infty)$ lattice.

The consequence: **the critical exponents of the 1D TFIM are identical to those of the 2D classical Ising model**. The exponent $\nu = 1$, $\beta = 1/8$, $\gamma = 7/4$ that we measured in notebook 02 all apply here. In notebook 07 we confirm this directly by applying the finite-size scaling collapse to the TFIM data.

In [ ]:
# Summary plot: phase diagram sketch from numerical data
# For several system sizes, plot the gap minimum location vs N
# It should converge to Gamma_c = J = 1 as N -> infinity.

N_pd_values  = [6, 8, 10, 12, 14, 16, 18, 20]
Gamma_dense  = np.linspace(0.5, 1.5, 80)
gap_min_locs = []

for N in N_pd_values:
    gaps_n = []
    for G in Gamma_dense:
        H     = build_tfim(N, J=J, Gamma=G, pbc=False)
        evals = np.sort(eigsh(H, k=2, which='SA', return_eigenvectors=False))
        gaps_n.append(evals[1] - evals[0])
    gap_min_locs.append(Gamma_dense[np.argmin(gaps_n)])
    print(f"N = {N:>2}: gap minimum at Gamma/J = {gap_min_locs[-1]:.4f}")

print(f"\nAll results should converge to Gamma_c / J = 1.0 as N -> inf.")

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(N_pd_values, gap_min_locs, 'o-', ms=7, lw=1.8, color='steelblue',
        label='Gap minimum location')
ax.axhline(1.0, color='k', ls='--', lw=1.2, label=r'$\Gamma_c / J = 1$ (exact)')
ax.set_xlabel('System size $N$', fontsize=13)
ax.set_ylabel(r'$\Gamma_{\rm min} / J$', fontsize=13)
ax.set_title('Convergence of the gap minimum to $\\Gamma_c$', fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Exercises

1. **Periodic boundary conditions.** Modify `build_tfim` to accept `pbc=True` and repeat the gap scaling at $\Gamma_c = J$. The critical exponent should still be $z = -1$, but the prefactor and finite-size corrections will differ. Compare the gap values at a given $N$ between OBC and PBC.

2. **Ground state wavefunction in the two limits.** For $N = 6$, $\Gamma = 0.05J$, extract the ground state eigenvector. Print the 5 basis states with the largest amplitudes $|\psi_s|^2$. Repeat for $\Gamma = 10J$. In the ordered limit you should see weight on $|{\uparrow\uparrow\uparrow\uparrow\uparrow\uparrow}\rangle$ and $|{\downarrow\downarrow\downarrow\downarrow\downarrow\downarrow}\rangle$; in the disordered limit all $2^N$ states should have approximately equal weight $1/2^N$.

3. **Longitudinal field.** Add a term $-h\sum_i \hat{\sigma}^z_i$ to the TFIM Hamiltonian (a field parallel to the Ising axis). This breaks the $\mathbb{Z}_2$ symmetry and turns the QPT into a smooth crossover, exactly as the external field does in the classical Ising model (notebook 04). Plot $m^2$ vs $\Gamma/J$ for $h = 0, 0.01, 0.05, 0.1$ and observe how the sharp transition is replaced by a crossover.

4. **Excitation gap in the disordered phase.** The exact result for the infinite chain is $\Delta = 2|\Gamma - J|$ near the transition. For $N = 20$ and $\Gamma/J \in [1.1, 2.0]$, plot the numerically computed gap against the exact expression. How large does $N$ need to be for the two to agree within 1%?